# 00. Torch GPU 셋업

이 문서는 PyTorch 실습에서 아래처럼 출력될 때 확인해야 할 내용을 정리한 가이드입니다.

```python
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)
```

만약 결과가 `Using device: cpu`라면, 현재 PyTorch가 GPU를 사용하지 못하고 있다는 뜻입니다.

## 목차

- [0-1. 먼저 알아야 할 핵심](#0-1.-먼저-알아야-할-핵심)
- [0-2. 왜 CPU로 잡히는가?](#0-2.-왜-CPU로-잡히는가)
- [0-3. GPU 사용 가능 조건](#0-3.-GPU-사용-가능-조건)
- [0-4. Windows에서 PyTorch GPU 환경 설치 순서](#0-4.-Windows에서-PyTorch-GPU-환경-설치-순서)
- [0-5. 설치 후 확인 코드](#0-5.-설치-후-확인-코드)
- [0-6. 자주 막히는 문제](#0-6.-자주-막히는-문제)
- [0-7. 정리](#0-7.-정리)


## 0-1. 먼저 알아야 할 핵심

`Using device: cpu`가 나온다고 해서 코드가 틀린 것은 아닙니다. 대부분은 다음 중 하나입니다.

- 노트북을 실행 중인 파이썬 환경에 **GPU 버전 PyTorch가 설치되지 않음**
- PC에 **CUDA 사용 가능한 NVIDIA GPU가 없음**
- **NVIDIA 드라이버**가 없거나 오래됨
- Jupyter가 내가 설치한 환경이 아니라 **다른 커널**로 실행 중임

즉, 코드 한 줄만 바꿔서 해결되는 문제가 아니라, **하드웨어 + 드라이버 + PyTorch 설치 환경 + Jupyter 커널**이 모두 맞아야 합니다.


## 0-2. 왜 CPU로 잡히는가?

이 코드는 GPU가 사용 가능할 때만 `cuda`를 선택합니다.

```python
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
```

여기서 `torch.cuda.is_available()`가 `False`이면 자동으로 `cpu`가 선택됩니다.

즉, 문제의 핵심은 다음 질문입니다.

**왜 `torch.cuda.is_available()`가 False인가?**


## 0-3. GPU 사용 가능 조건

PyTorch에서 일반적으로 GPU 학습을 하려면 아래 조건이 필요합니다.

1. **NVIDIA GPU가 있어야 함**
2. **NVIDIA 드라이버가 정상 설치되어 있어야 함**
3. **CUDA 지원 PyTorch가 설치되어 있어야 함**
4. **Jupyter Notebook이 그 PyTorch가 설치된 환경으로 실행되어야 함**

중요한 점은, 보통 PyTorch 실습에서 말하는 `cuda`는 NVIDIA CUDA 기준입니다. AMD 내장 그래픽이나 일반 CPU만으로는 `torch.cuda.is_available()`가 True가 되지 않습니다.


## 0-4. Windows에서 PyTorch GPU 환경 설치 순서

아래 순서대로 확인하는 것이 가장 안전합니다.


### 1) 내 PC에 NVIDIA GPU가 있는지 확인

Windows 터미널이나 명령 프롬프트에서 다음 명령을 실행합니다.

```powershell
nvidia-smi
```

정상이라면 GPU 이름, 드라이버 버전, CUDA 관련 정보가 출력됩니다.

만약 `nvidia-smi` 명령이 없거나 에러가 난다면 다음 가능성이 큽니다.

- NVIDIA GPU가 없음
- NVIDIA 드라이버가 설치되지 않음
- 드라이버가 깨졌거나 PATH 문제


### 2) 가상환경을 따로 만드는 것을 권장

실습용 환경을 분리해두면 CPU 버전과 GPU 버전이 섞이는 문제를 줄일 수 있습니다.

```powershell
python -m venv .venv
.\.venv\Scripts\activate
python -m pip install --upgrade pip
```


### 3) GPU 지원 PyTorch 설치

가장 중요한 단계입니다. CPU 전용 PyTorch가 설치되어 있으면 `torch.cuda.is_available()`는 계속 `False`입니다.

아래 명령은 예시입니다.

```powershell
pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124
```

주의할 점:

- `cu124`는 CUDA 12.4 계열용 wheel 예시입니다.
- 시점에 따라 PyTorch 권장 버전은 바뀔 수 있습니다.
- 가장 정확한 설치 명령은 **PyTorch 공식 홈페이지의 설치 가이드**에서 현재 기준으로 확인하는 것이 좋습니다.

이미 CPU 버전이 꼬여 있다면 아래처럼 재설치하는 편이 깔끔합니다.

```powershell
pip uninstall -y torch torchvision torchaudio
pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124
```


### 4) Jupyter 커널도 같은 환경으로 연결

PyTorch를 GPU 버전으로 잘 설치해도, Jupyter가 다른 파이썬 환경을 보고 있으면 노트북에서는 계속 CPU로 나올 수 있습니다.

같은 가상환경에서 아래 명령을 실행합니다.

```powershell
pip install ipykernel notebook
python -m ipykernel install --user --name deeplearning-gpu --display-name "Python (deeplearning-gpu)"
```

그 다음 Jupyter Notebook에서 커널을 `Python (deeplearning-gpu)`로 바꿉니다.


### 5) 설치 확인

노트북에서 아래 코드를 실행합니다.


In [ ]:
import torch

print('torch version:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
print('cuda device count:', torch.cuda.device_count())

if torch.cuda.is_available():
    print('current device:', torch.cuda.current_device())
    print('device name:', torch.cuda.get_device_name(0))
    device = torch.device('cuda')
else:
    device = torch.device('cpu')

print('Using device:', device)


정상적인 경우 예를 들어 다음처럼 나옵니다.

```text
torch version: 2.x.x
cuda available: True
cuda device count: 1
current device: 0
device name: NVIDIA GeForce ...
Using device: cuda
```


## 0-5. 설치 후 확인 코드

아래 코드는 실제로 텐서를 GPU에 올려보는 최소 테스트입니다.


In [ ]:
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
x = torch.tensor([1.0, 2.0, 3.0]).to(device)

print('Using device:', device)
print(x)


출력 예시:

```text
Using device: cuda
tensor([1., 2., 3.], device='cuda:0')
```

여기서 텐서 출력에 `device='cuda:0'`가 보이면 GPU 사용이 실제로 되고 있는 것입니다.


## 0-6. 자주 막히는 문제

### 문제 1) `torch.cuda.is_available()`가 계속 False

가능한 원인:

- CPU 전용 PyTorch 설치
- NVIDIA 드라이버 미설치
- Jupyter 커널이 다른 환경을 사용 중
- NVIDIA GPU가 없는 PC


### 문제 2) 터미널에서는 되는데 노트북에서는 안 됨

이 경우는 거의 커널 문제입니다.

아래 두 위치에서 같은 파이썬을 보고 있는지 확인해야 합니다.

- 터미널에서의 `python`
- Jupyter Notebook 커널의 파이썬

노트북에서 다음 코드를 실행해서 확인할 수 있습니다.

```python
import sys
print(sys.executable)
```


### 문제 3) 내 PC가 노트북인데 GPU가 있는지 모르겠음

작업 관리자(Task Manager)에서 `성능` 탭을 보면 GPU가 표시됩니다. 또는 아래 명령으로 확인할 수 있습니다.

```powershell
nvidia-smi
```

아예 NVIDIA GPU가 없다면, 로컬에서는 `cuda` 사용이 불가능합니다. 이 경우에는:

- CPU로 실습을 진행하거나
- Google Colab 같은 클라우드 GPU 환경을 사용하거나
- GPU가 있는 데스크톱/서버 환경을 사용하는 방법이 있습니다.


### 문제 4) 코드에서 따로 수정할 것은 없는가?

기본적으로는 아래 코드면 충분합니다.

```python
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
images = images.to(device)
labels = labels.to(device)
```

즉, 코드보다는 환경 구성이 더 중요합니다.


## 0-7. 정리

`Using device: cpu`를 `Using device: cuda`로 바꾸려면 보통 아래 순서로 확인하면 됩니다.

1. 내 PC에 NVIDIA GPU가 있는지 확인
2. `nvidia-smi`가 정상 동작하는지 확인
3. GPU 지원 PyTorch를 설치
4. Jupyter 커널을 그 환경으로 연결
5. `torch.cuda.is_available()`와 `torch.cuda.get_device_name(0)`로 최종 확인

다음 실습 노트북을 실행하기 전에 이 문서의 확인 코드를 먼저 돌려보면, 왜 CPU로 잡히는지 빠르게 판단할 수 있습니다.
